# Deep SVDD x64 Baseline

This notebook is the curated submission-facing entry point for the Deep SVDD baseline on the shared `x64` anomaly benchmark.
It defaults to reusing the saved checkpoint and evaluation artifacts instead of retraining.


## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/svdd/x64/baseline/train_config.toml`
- Training script: `scripts/train_svdd.py`
- Artifact root: `experiments/anomaly_detection/svdd/x64/baseline/artifacts/svdd_baseline/`
- Default behavior: reuse `best_model.pt`, `history.json`, and `results/evaluation/` outputs when they already exist


### Setup And Imports

This cell resolves the repo root, exposes `src/` on `PYTHONPATH`, and imports the shared utilities used for loading configs, datasets, and the SVDD model.


In [ ]:
from __future__ import annotations
import json
import os
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch
import tomllib
try:
    from IPython.display import display
except ImportError:

    def display(obj: object) -> None:
        print(obj)
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'wafer_defect').exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists():
            REPO_ROOT = candidate
            break
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset


### Load The Local Config And Run Controls

This cell points the notebook at the local experiment config and exposes the main rerun flags. Leave both flags as `False` for the normal artifact-first submission workflow.


In [ ]:
CONFIG_PATH = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'svdd' / 'x64' / 'baseline' / 'train_config.toml'
config = load_toml(CONFIG_PATH)
OUTPUT_DIR = REPO_ROOT / config['run']['output_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR = OUTPUT_DIR / 'results/evaluation'
PLOTS_DIR = OUTPUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RETRAIN = False
RERUN_EVALUATION = False
TOP_K = 8
display(config)


### Preview The Curated Benchmark Split

This cell confirms that the notebook is using the processed `50k / 5%` metadata and shows the split counts that the model will use.


In [ ]:
metadata_path = REPO_ROOT / config['data']['metadata_csv']
metadata_df = pd.read_csv(metadata_path)
split_summary_df = metadata_df.groupby(['split', 'is_anomaly']).size().reset_index(name='count').sort_values(['split', 'is_anomaly']).reset_index(drop=True)
train_dataset = WaferMapDataset(metadata_path, split='train', image_size=int(config['data']['image_size']))
val_dataset = WaferMapDataset(metadata_path, split='val', image_size=int(config['data']['image_size']))
test_dataset = WaferMapDataset(metadata_path, split='test', image_size=int(config['data']['image_size']))
print(f'Metadata CSV: {metadata_path}')
display(split_summary_df)
display(metadata_df.head(5))


### Train Or Reuse Saved Artifacts

This cell either reuses the saved training artifacts or runs `scripts/train_svdd.py` with the local config. The default path is to skip retraining and load the existing history and summary files.


In [ ]:
history_path = OUTPUT_DIR / 'results' / 'history.json'
summary_path = OUTPUT_DIR / 'results' / 'summary.json'
best_model_path = OUTPUT_DIR / 'checkpoints' / 'best_model.pt'
artifacts_ready = history_path.exists() and summary_path.exists() and best_model_path.exists()
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC_ROOT) if not env.get('PYTHONPATH') else str(SRC_ROOT) + os.pathsep + env['PYTHONPATH']
if RETRAIN:
    train_cmd = [sys.executable, 'scripts/train_svdd.py', '--config', str(CONFIG_PATH.relative_to(REPO_ROOT))]
    print('Running:', ' '.join(train_cmd))
    subprocess.run(train_cmd, cwd=REPO_ROOT, env=env, check=True)
elif artifacts_ready:
    print(f'Found existing training artifacts in {OUTPUT_DIR}. Skipping retraining.')
else:
    print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
history = json.loads(history_path.read_text(encoding='utf-8'))
history_df = pd.DataFrame(history)
training_summary = json.loads(summary_path.read_text(encoding='utf-8'))
display(pd.DataFrame([training_summary]))


### Plot The Training Curve

This cell rebuilds the SVDD training plot from `history.json`, shows it inline, and saves it into the local `plots/` folder.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.plot(history_df['epoch'], history_df['train_loss'], label='train')
ax.plot(history_df['epoch'], history_df['val_loss'], label='val')
ax.set_title('SVDD Distance Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
fig.savefig(PLOTS_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)


### Evaluate Or Reuse Saved Scores

This cell reuses the saved score CSVs and summary when they already exist. If the evaluation cache is missing, it reruns the shared evaluation script against the saved `best_model.pt` checkpoint.


In [ ]:
evaluation_summary_path = EVALUATION_DIR / 'summary.json'
val_scores_path = EVALUATION_DIR / 'val_scores.csv'
test_scores_path = EVALUATION_DIR / 'test_scores.csv'
threshold_sweep_path = EVALUATION_DIR / 'threshold_sweep.csv'
evaluation_ready = all((path.exists() for path in [evaluation_summary_path, val_scores_path, test_scores_path, threshold_sweep_path]))
if RERUN_EVALUATION:
    eval_cmd = [sys.executable, 'scripts/evaluate_reconstruction_model.py', '--checkpoint', str(best_model_path.relative_to(REPO_ROOT)), '--config', str(CONFIG_PATH.relative_to(REPO_ROOT)), '--output-dir', str(EVALUATION_DIR.relative_to(REPO_ROOT))]
    print('Running:', ' '.join(eval_cmd))
    subprocess.run(eval_cmd, cwd=REPO_ROOT, env=env, check=True)
elif evaluation_ready:
    print(f'Found existing evaluation outputs in {EVALUATION_DIR}. Skipping rerun.')
else:
    print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
evaluation_summary = json.loads(evaluation_summary_path.read_text(encoding='utf-8'))
val_scores_df = pd.read_csv(val_scores_path)
test_scores_df = pd.read_csv(test_scores_path)
threshold_sweep_df = pd.read_csv(threshold_sweep_path)
threshold = float(evaluation_summary['threshold'])
best_sweep = dict(evaluation_summary.get('best_threshold_sweep', {}))
display(pd.DataFrame([evaluation_summary['metrics_at_validation_threshold']]))
display(pd.DataFrame([best_sweep]))


### Plot Score Distributions And Threshold Sweep

This cell recreates the main evaluation figures from the saved score CSVs and saves the combined plot to the local artifact folder.


In [ ]:
focus_low = float(threshold_sweep_df['threshold'].quantile(0.01))
focus_high = float(threshold_sweep_df['threshold'].quantile(0.99))
focus_low = min(focus_low, threshold, float(best_sweep['threshold']))
focus_high = max(focus_high, threshold, float(best_sweep['threshold']))
focus_pad = max((focus_high - focus_low) * 0.1, 1e-06)
focus_low = max(0.0, focus_low - focus_pad)
focus_high = focus_high + focus_pad
cm = evaluation_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])
cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.hist(test_scores_df.loc[test_scores_df['is_anomaly'] == 0, 'score'], bins=30, alpha=0.7, label='normal')
ax.hist(test_scores_df.loc[test_scores_df['is_anomaly'] == 1, 'score'], bins=30, alpha=0.7, label='anomaly')
ax.axvline(threshold, color='red', linestyle='--', label=f'val threshold = {threshold:.2e}')
ax.set_title('SVDD Test Score Distribution')
ax.set_xlabel('L2 distance from SVDD center')
ax.set_ylabel('Count')
ax.legend()
fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
ax.axvline(threshold, color='red', linestyle='--', label='val threshold')
ax.axvline(float(best_sweep['threshold']), color='green', linestyle=':', label='best sweep')
ax.set_title('SVDD Threshold Sweep (Full)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Metric value')
ax.legend()
fig.savefig(PLOTS_DIR / 'threshold_sweep.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
ax.axvline(threshold, color='red', linestyle='--', label='val threshold')
ax.axvline(float(best_sweep['threshold']), color='green', linestyle=':', label='best sweep')
ax.set_xlim(focus_low, focus_high)
ax.set_title('SVDD Threshold Sweep (Zoomed)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Metric value')
ax.legend()
fig.savefig(PLOTS_DIR / 'threshold_sweep_zoomed.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
heatmap = ax.imshow(cm_df.to_numpy(), cmap='Blues')
ax.set_xticks(range(cm_df.shape[1]), cm_df.columns)
ax.set_yticks(range(cm_df.shape[0]), cm_df.index)
ax.set_title('SVDD Confusion Matrix')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
for row_idx in range(cm_df.shape[0]):
    for col_idx in range(cm_df.shape[1]):
        ax.text(col_idx, row_idx, str(int(cm_df.iat[row_idx, col_idx])), ha='center', va='center', color='black', fontsize=12)
fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
fig.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)
display(cm_df)
display(threshold_sweep_df.sort_values('f1', ascending=False).head(10))


### Show The Highest-Scored Test Examples

This cell uses the saved `sample_index` column from the evaluation CSV to display the top-ranked SVDD anomalies and saves the qualitative figure to the artifact folder.


In [ ]:
ranked_test_scores_df = test_scores_df.sort_values('score', ascending=False).reset_index(drop=True)
top_k = min(TOP_K, len(ranked_test_scores_df))
rows = 2
cols = max(1, top_k // 2)
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), constrained_layout=True)
axes = axes.ravel() if hasattr(axes, 'ravel') else [axes]
for idx in range(top_k):
    row = ranked_test_scores_df.iloc[idx]
    wafer_map, label = test_dataset[int(row['sample_index'])]
    axes[idx].imshow(wafer_map[0], cmap='gray')
    axes[idx].set_title(f"score={row['score']:.4f}\nlabel={int(label)}")
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])
for idx in range(top_k, len(axes)):
    axes[idx].axis('off')
fig.savefig(PLOTS_DIR / 'top_scored_examples.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)


---

## Holdout Evaluation: Expanded 70k Normal / 3.5k Defect Test Set

This section evaluates the saved SVDD checkpoint on the secondary holdout split.
The holdout uses the same 5k validation normals for threshold derivation but replaces
the test set with **70,000 normal + 3,500 defect** wafers.

If cached holdout results already exist they are loaded without re-running the script.


In [ ]:
RUN_HOLDOUT_EVALUATION = False
FORCE_HOLDOUT_RERUN = False
HOLDOUT_METADATA_PATH = REPO_ROOT / 'data/processed/x64/wm811k/metadata_50k_5pct_holdout70k_3p5k.csv'
HOLDOUT_OUTPUT_DIR = OUTPUT_DIR / 'holdout70k_3p5k'
print(f'Holdout metadata: {HOLDOUT_METADATA_PATH}')
print(f'Exists:           {HOLDOUT_METADATA_PATH.exists()}')
print(f'Output dir:       {HOLDOUT_OUTPUT_DIR}')


In [ ]:
if not RUN_HOLDOUT_EVALUATION:
    print('Holdout evaluation skipped.')
else:
    HOLDOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    _h_summary_path = HOLDOUT_OUTPUT_DIR / 'summary.json'
    _h_val_scores = HOLDOUT_OUTPUT_DIR / 'val_scores.csv'
    _h_test_scores = HOLDOUT_OUTPUT_DIR / 'test_scores.csv'
    _h_sweep = HOLDOUT_OUTPUT_DIR / 'threshold_sweep.csv'
    _h_required = [_h_summary_path, _h_val_scores, _h_test_scores, _h_sweep]
    if not FORCE_HOLDOUT_RERUN and all((p.exists() for p in _h_required)):
        print(f'Loaded cached holdout results from {HOLDOUT_OUTPUT_DIR}')
        print('Set FORCE_HOLDOUT_RERUN = True to re-evaluate.')
    elif FORCE_HOLDOUT_RERUN:
        if FORCE_HOLDOUT_RERUN:
            holdout_cmd = [sys.executable, 'scripts/evaluate_reconstruction_model.py', '--checkpoint', str(best_model_path.relative_to(REPO_ROOT)), '--config', str(CONFIG_PATH.relative_to(REPO_ROOT)), '--metadata-csv', str(HOLDOUT_METADATA_PATH.relative_to(REPO_ROOT)), '--output-dir', str(HOLDOUT_OUTPUT_DIR.relative_to(REPO_ROOT))]
            print('Running:', ' '.join(holdout_cmd))
            subprocess.run(holdout_cmd, cwd=REPO_ROOT, env=env, check=True)
            print(f'Holdout results saved to {HOLDOUT_OUTPUT_DIR}')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    _h_summary = json.loads(_h_summary_path.read_text(encoding='utf-8'))
    _h_val_df = pd.read_csv(_h_val_scores)
    _h_test_df = pd.read_csv(_h_test_scores)
    _h_sweep_df = pd.read_csv(_h_sweep)
    _h_threshold = float(_h_summary['threshold'])
    _h_metrics = _h_summary['metrics_at_validation_threshold']
    _h_best = _h_summary.get('best_threshold_sweep', {})
    print(f'\n-- Holdout Results (70k normal / 3.5k defect) --')
    print(f'  Threshold:           {_h_threshold:.8f}')
    print(f"  Precision:           {_h_metrics['precision']:.6f}")
    print(f"  Recall:              {_h_metrics['recall']:.6f}")
    print(f"  F1:                  {_h_metrics['f1']:.6f}")
    print(f"  AUROC:               {_h_metrics['auroc']:.6f}")
    print(f"  AUPRC:               {_h_metrics['auprc']:.6f}")
    print(f"  Predicted anomalies: {_h_metrics.get('predicted_anomalies', 'n/a')}")
    if _h_best:
        print(f"  Best sweep F1:       {float(_h_best.get('f1', 0)):.6f}")
    _holdout_meta = pd.read_csv(HOLDOUT_METADATA_PATH)
    _test_meta = _holdout_meta[_holdout_meta['split'] == 'test'].reset_index(drop=True).copy()
    _test_meta['score'] = _h_test_df['score'].values
    _test_meta['predicted'] = (_test_meta['score'] >= _h_threshold).astype(int)
    _defect_col = next((c for c in ['defect_type', 'failureType'] if c in _test_meta.columns), None)
    if _defect_col:
        _h_defect = _test_meta[_test_meta['is_anomaly'] == 1].groupby(_defect_col).apply(lambda g: pd.Series({'count': len(g), 'detected': int(g['predicted'].sum()), 'recall': g['predicted'].mean()})).reset_index().sort_values('recall')
        print(f'\n-- Per-Defect Recall --')
        display(_h_defect)
        _h_defect.to_csv(HOLDOUT_OUTPUT_DIR / 'defect_recall.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(_h_test_df[_h_test_df['is_anomaly'] == 0]['score'], bins=50, alpha=0.7, label='normal')
    axes[0].hist(_h_test_df[_h_test_df['is_anomaly'] == 1]['score'], bins=50, alpha=0.7, label='anomaly')
    axes[0].axvline(_h_threshold, color='red', linestyle='--', label=f'threshold={_h_threshold:.2e}')
    axes[0].set_title('Holdout Score Distribution')
    axes[0].set_xlabel('L2 distance from SVDD center')
    axes[0].legend()
    axes[1].plot(_h_sweep_df['threshold'], _h_sweep_df['precision'], label='precision')
    axes[1].plot(_h_sweep_df['threshold'], _h_sweep_df['recall'], label='recall')
    axes[1].plot(_h_sweep_df['threshold'], _h_sweep_df['f1'], label='f1')
    axes[1].axvline(_h_threshold, color='red', linestyle='--', label='val threshold')
    axes[1].set_title('Holdout Threshold Sweep')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(HOLDOUT_OUTPUT_DIR / 'holdout_distribution_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plot saved to {HOLDOUT_OUTPUT_DIR / 'holdout_distribution_sweep.png'}")
